<a href="https://colab.research.google.com/github/WenhuiCaii/SEED/blob/main/Streetview_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install mit_semseg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 1.3 MB/s eta 0:00:00


In [ ]:
# import required packages
import os, csv, torch, scipy.io, PIL.Image, torchvision.transforms
import numpy as np
from mit_semseg.models import ModelBuilder, SegmentationModule
from mit_semseg.utils import colorEncode
import pandas as pd
import pickle

In [ ]:
import zipfile
zip_ref = zipfile.ZipFile("/content/drive/MyDrive/Streetview/Replicates_deleted.zip", 'r')
zip_ref.extractall("/content/drive/MyDrive/Streetview/")
zip_ref.close()

KeyboardInterrupt: 

KeyboardInterrupt: 

In [ ]:
# Loading image classification

colors = scipy.io.loadmat('/content/color150.mat')['colors']
names = {}
with open('/content/object150_info.csv') as f:
    reader = csv.reader(f)
    next(reader)
    for row in reader:
        names[int(row[0])] = row[5].split(";")[0]

def visualize_result(pred, index=None):
    # filter prediction class if requested
    if index is not None:
        pred = pred.copy()
        pred[pred != index] = -1
        print(f'{names[index+1]}:')

    # colorize prediction
    pred_color = colorEncode(pred, colors).astype(np.uint8)

    # aggregate images and save
    #im_vis = np.concatenate((img, pred_color), axis=1)
    #display(PIL.Image.fromarray(im_vis))
    return pred_color

In [ ]:
# Network Builders
net_encoder = ModelBuilder.build_encoder(
    arch='resnet50dilated',
    fc_dim=2048,
    weights='/content/encoder_epoch_20.pth')
net_decoder = ModelBuilder.build_decoder(
    arch='ppm_deepsup',
    fc_dim=2048,
    num_class=150,
    weights='/content/decoder_epoch_20.pth',
    use_softmax=True)

crit = torch.nn.NLLLoss(ignore_index=-1)
segmentation_module = SegmentationModule(net_encoder, net_decoder, crit)
segmentation_module.eval()
segmentation_module.cpu()

Loading weights for net_encoder


RuntimeError: unexpected EOF, expected 9055554 more bytes. The file might be corrupted.

In [ ]:
lst = os.listdir('/content/streetview')
semantic = pd.DataFrame()
for i in range(len(lst)):
  # Load and normalize one image as a singleton tensor batch
  pil_to_tensor = torchvision.transforms.Compose([
      torchvision.transforms.ToTensor(),
      torchvision.transforms.Normalize(
          mean=[0.485, 0.456, 0.406], # These are RGB mean+std values
          std=[0.229, 0.224, 0.225])  # across a large photo dataset.
      ])
  pil_image = PIL.Image.open('/content/streetview/'+lst[i]).convert('RGB')
  # pil_image = PIL.Image.open('demo_pic/demo_pic.png').convert('RGB')
  img_original = np.array(pil_image)
  img_data = pil_to_tensor(pil_image)
  singleton_batch = {'img_data': img_data[None].cpu()}
  output_size = img_data.shape[1:]
  with torch.no_grad():
    scores = segmentation_module(singleton_batch, segSize=output_size)
    # Get the predicted scores for each pixel
  _, pred = torch.max(scores, dim=1)
  pred = pred.cpu()[0].np()
  # loop to calculate class index
  semantic['id'] = lst[i]
  for j in range(150):
    vs = visualize_result(pred, j)
    semantic[j] = np.count_nonzero((vs != [0, 0, 0]).all(axis = 2)) / (vs.shape[0] * vs.shape[1]) # ratio

In [ ]:
import pandas as pd
seg = pd.read_csv('/content/semantic_1.csv')

In [ ]:
pred_X = seg.iloc[:,2:152]
pred_X

,0,1,10,100,101,102,103,104,105,106,...,90,91,92,93,94,95,96,97,98,99
0,0.032986,0.000000,0.0,0,0,0,0,0.0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0.016140,0.042372,0.0,0,0,0,0,0.0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0.000597,0.025743,0.0,0,0,0,0,0.0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0.000000,0.024577,0.0,0,0,0,0,0.0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0.061707,0.000000,0.0,0,0,0,0,0.0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109314,0.018663,0.000000,0.0,0,0,0,0,0.0,0,0,...,0,0,0,0,0,0,0,0,0,0
109315,0.000000,0.009196,0.0,0,0,0,0,0.0,0,0,...,0,0,0,0,0,0,0,0,0,0
109316,0.000000,0.000000,0.0,0,0,0,0,0.0,0,0,...,0,0,0,0,0,0,0,0,0,0
109317,0.000298,0.452582,0.0,0,0,0,0,0.0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
! pip install scikit-learn==1.1.1

In [ ]:
# Indices prediction using US models
import pickle
out_model = '/content/model_beautiful.pkl'
with open(out_model,'rb') as fl:
    clf2 = pickle.load(fl)

In [ ]:
pred_X = seg.iloc[:,2:152].values
clf2.predict(pred_X[:10])
clf2.predict_proba(pred_X[:10])
scores = clf2.predict_proba(pred_X[:])[:,-1]
seg['beautiful'] = scores
seg.to_csv('/content/semantic_beautiful.csv')

In [ ]:
# Indices prediction using US models
import pickle
out_model = '/content/model_boring.pkl'
with open(out_model,'rb') as fl:
    clf2 = pickle.load(fl)

In [ ]:
pred_X = seg.iloc[:,2:152].values
clf2.predict(pred_X[:10])
clf2.predict_proba(pred_X[:10])
scores = clf2.predict_proba(pred_X[:])[:,-1]
seg['boring'] = scores
seg.to_csv('/content/semantic_boring.csv')

In [ ]:
import pickle
out_model = '/content/model_depressing.pkl'
with open(out_model,'rb') as fl:
    clf2 = pickle.load(fl)

In [ ]:
pred_X = seg.iloc[:,2:152].values
clf2.predict(pred_X[:10])
clf2.predict_proba(pred_X[:10])
scores = clf2.predict_proba(pred_X[:])[:,-1]
seg['depressing'] = scores
seg.to_csv('/content/semantic_depressing.csv')

In [ ]:
import pickle
out_model = '/content/model_livelier.pkl'
with open(out_model,'rb') as fl:
    clf2 = pickle.load(fl)

In [ ]:
pred_X = seg.iloc[:,2:152].values
clf2.predict(pred_X[:10])
clf2.predict_proba(pred_X[:10])
scores = clf2.predict_proba(pred_X[:])[:,-1]
seg['livelier'] = scores
seg.to_csv('/content/semantic_livelier.csv')

In [ ]:
import pickle
out_model = '/content/model_safer.pkl'
with open(out_model,'rb') as fl:
    clf2 = pickle.load(fl)

In [ ]:
pred_X = seg.iloc[:,2:152].values
clf2.predict(pred_X[:10])
clf2.predict_proba(pred_X[:10])
scores = clf2.predict_proba(pred_X[:])[:,-1]
seg['safer'] = scores
seg.to_csv('/content/semantic_safer.csv')

In [ ]:
import pickle
out_model = '/content/model_wealthier.pkl'
with open(out_model,'rb') as fl:
    clf2 = pickle.load(fl)

In [ ]:
pred_X = seg.iloc[:,2:152].values
clf2.predict(pred_X[:10])
clf2.predict_proba(pred_X[:10])
scores = clf2.predict_proba(pred_X[:])[:,-1]
seg['wealthier'] = scores
seg.to_csv('/content/semantic_wealthier.csv')